# 技能4 · Day 4 上机：平台战略 + 生态设计

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **networkx** 构建真实平台生态网络（平台/开发者/消费者/互补者），计算网络效应指标
2. 用 **networkx** 图算法分析度分布/聚类系数/核心-边缘结构
3. 用 **pandas** 量化多归属率（multi-homing）和锁定度（lock-in index）
4. 用 **matplotlib** 可视化平台生态网络拓扑和核心-边缘结构
5. 用 **pandas + numpy** 构建平台战略框架（网络效应强度/赢者通吃倾向）
6. 用 **numpy 蒙特卡洛模拟 + 贝叶斯推断** 实现天道推演--平台临界点仿真

## 真实数据
- App Store（30%抽成，~180万应用）、Google Play（30%抽成，~250万应用）
- Hugging Face（0%抽成，~100万模型/20万数据集/40万Spaces）
- MCP Ecosystem（0%抽成，开放协议，~5000工具）
- 参与者：Meta/Google/Anthropic/OpenAI/Microsoft 等真实公司及其平台归属


## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> networkx/matplotlib/pandas/numpy 是纯 Python 库，无需外部服务。

In [ ]:
# !pip install networkx matplotlib pandas numpy -q

import warnings
warnings.filterwarnings('ignore')

import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print(f'networkx {nx.__version__}, pandas {pd.__version__}, numpy {np.__version__}')
print('Libraries loaded successfully.')


## 1. 数据背景与营销映射

**构建对象**：真实 AI 平台生态网络，覆盖4类参与者节点和6类关系边：

| 参与者类型 | 示例 | 属性 |
|-----------|------|------|
| Platform | App Store, Google Play, Hugging Face, MCP Ecosystem | commission, apps/models, launched |
| Developer | Meta, Google, Mistral AI, Apple, Anthropic, OpenAI, Microsoft | - |
| Consumer | Enterprise Users, Individual Users, Research Labs, AI Startups | - |
| Complementor | LangChain, Weights & Biases, vLLM, FastAPI, PyTorch | - |

| 关系类型 | 示例 | 含义 |
|---------|------|------|
| PUBLISHES_ON | Meta -> Hugging Face | 开发者在平台发布 |
| USES | Enterprise Users -> Hugging Face | 消费者使用平台 |
| INTEGRATES_WITH | LangChain -> Hugging Face | 互补者与平台集成 |
| DEPENDS_ON | vLLM -> Hugging Face | 互补者依赖平台 |
| COMPETES_WITH | App Store <-> Google Play | 平台间竞争 |
| COMPLEMENTS | Hugging Face <-> MCP Ecosystem | 平台间互补 |

**营销映射**：AI 营销平台生态连接广告主（需求方）/创作者（供给方）/数据提供方/MCP 工具开发者，
本 Day 的生态网络分析方法可直接迁移到营销 Agent 平台生态分析。

## TODO 1：用 networkx 构建平台生态网络

**核心任务**：创建一个 MultiDiGraph，添加4个真实平台节点（App Store/Google Play/Hugging Face/MCP Ecosystem），
12个开发者节点、5个消费者节点、5个互补者节点，以及 PUBLISHES_ON/USES/INTEGRATES_WITH/DEPENDS_ON/COMPETES_WITH/COMPLEMENTS 关系边。

**提示**：
- `G.add_node(name, node_type=..., commission=..., apps=...)` 添加节点
- `G.add_edge(src, dst, relation=...)` 添加边
- 平台数据：App Store 30%抽成/180万应用，Google Play 30%/250万，HF 0%/100万模型，MCP 0%/5000工具
- 开发者归属：Meta->HF, Google->HF+GP, Mistral->HF, Apple->AS, Anthropic->MCP, OpenAI->MCP,
  Microsoft->HF+MCP, Indie_Dev_A->AS+GP, Indie_Dev_B->GP+HF, Indie_Dev_C->AS,
  Community_Dev_D->HF+MCP, Community_Dev_E->MCP+HF
- 消费者：Enterprise_Users->HF+MCP, Individual_Users->AS+GP, Research_Labs->HF,
  AI_Startups->MCP+HF+GP, Mobile_Users->AS+GP
- 互补者：LangChain->HF+MCP, Weights_Biases->HF, vLLM->HF, FastAPI->MCP, PyTorch->HF
- 依赖边：LangChain->HF, vLLM->HF, FastAPI->MCP, PyTorch->HF
- 竞争/互补：AS<->GP(COMPETES_WITH), HF<->MCP(COMPLEMENTS)

In [ ]:
# 构建平台生态网络
G = nx.MultiDiGraph()

# 添加4个平台节点（真实公开数据：抽成比例/生态规模/启动年份）
platform_data = {
    "App Store":      {"commission": 0.30, "apps": 1_800_000,  "developers": 34000000, "launched": 2008},
    "Google Play":    {"commission": 0.30, "apps": 2_500_000,  "developers": 25000000, "launched": 2008},
    "Hugging Face":   {"commission": 0.00, "models": 1_000_000, "datasets": 200000, "spaces": 400000, "launched": 2016},
    "MCP Ecosystem":  {"commission": 0.00, "tools": 5000,      "servers": 1200, "launched": 2024},
}
for name, attrs in platform_data.items():
    G.add_node(name, node_type="platform", **attrs)

# 添加12个开发者节点（真实公司及其平台归属）
developers = [
    ("Meta",           ["Hugging Face"]),
    ("Google",         ["Hugging Face", "Google Play"]),
    ("Mistral AI",     ["Hugging Face"]),
    ("Apple",          ["App Store"]),
    ("Anthropic",      ["MCP Ecosystem"]),
    ("OpenAI",         ["MCP Ecosystem"]),
    ("Microsoft",      ["Hugging Face", "MCP Ecosystem"]),
    ("Indie_Dev_A",    ["App Store", "Google Play"]),
    ("Indie_Dev_B",    ["Google Play", "Hugging Face"]),
    ("Indie_Dev_C",    ["App Store"]),
    ("Community_Dev_D", ["Hugging Face", "MCP Ecosystem"]),
    ("Community_Dev_E", ["MCP Ecosystem", "Hugging Face"]),
]
for name, platforms_list in developers:
    G.add_node(name, node_type="developer")
    for p in platforms_list:
        G.add_edge(name, p, relation="PUBLISHES_ON")

# 添加5个消费者节点
consumers = [
    ("Enterprise_Users",  ["Hugging Face", "MCP Ecosystem"]),
    ("Individual_Users",  ["App Store", "Google Play"]),
    ("Research_Labs",     ["Hugging Face"]),
    ("AI_Startups",       ["MCP Ecosystem", "Hugging Face", "Google Play"]),
    ("Mobile_Users",      ["App Store", "Google Play"]),
]
for name, platforms_list in consumers:
    G.add_node(name, node_type="consumer")
    for p in platforms_list:
        G.add_edge(name, p, relation="USES")

# 添加5个互补者节点
complementors = [
    ("LangChain",       ["Hugging Face", "MCP Ecosystem"]),
    ("Weights_Biases",  ["Hugging Face"]),
    ("vLLM",            ["Hugging Face"]),
    ("FastAPI",         ["MCP Ecosystem"]),
    ("PyTorch",         ["Hugging Face"]),
]
for name, platforms_list in complementors:
    G.add_node(name, node_type="complementor")
    for p in platforms_list:
        G.add_edge(name, p, relation="INTEGRATES_WITH")

# 添加依赖边（DEPENDS_ON）
G.add_edge("LangChain", "Hugging Face", relation="DEPENDS_ON")
G.add_edge("vLLM", "Hugging Face", relation="DEPENDS_ON")
G.add_edge("FastAPI", "MCP Ecosystem", relation="DEPENDS_ON")
G.add_edge("PyTorch", "Hugging Face", relation="DEPENDS_ON")

# 添加竞争/互补边
G.add_edge("App Store", "Google Play", relation="COMPETES_WITH")
G.add_edge("Hugging Face", "MCP Ecosystem", relation="COMPLEMENTS")

# 测试
print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
node_types = nx.get_node_attributes(G, 'node_type')
print(f'\nNode type distribution:')
print(pd.Series(node_types).value_counts())
edge_relations = [d['relation'] for _, _, d in G.edges(data=True)]
print(f'\nEdge relation distribution:')
print(pd.Series(edge_relations).value_counts())


## 2. 网络效应指标：图算法的真正价值

networkx 的图算法让我们能量化平台生态的拓扑特征：

| 指标 | 算法 | 平台战略含义 |
|------|------|-------------|
| 度分布 | `G.in_degree()` / `G.out_degree()` | 平台的连接数=生态影响力 |
| 聚类系数 | `nx.clustering()` | 参与者间的聚集=生态密度 |
| 核心-边缘 | `nx.core_number()` | 谁在生态核心，谁在边缘 |

**核心-边缘结构的战略含义**：核心节点是生态的枢纽--如果移除，网络会碎片化。
边缘节点是可替代的参与者。平台战略的核心目标之一是让自己成为生态的核心节点。


## TODO 2：网络效应指标计算（度分布/聚类系数/核心-边缘）

**核心任务**：用 networkx 图算法计算平台生态网络的结构指标。
- 计算入度/出度分布（均值/最大值/最小值）
- 计算聚类系数（转换为无向图后，带权重）
- 用 core_number 识别核心-边缘结构
- 打印 Top 5 聚类系数节点和核心/边缘节点列表

**提示**：
- `nx.Graph()` 从 MultiDiGraph 转换为无向图（合并多重边为权重）
- `nx.clustering(G_undirected, weight='weight')` 带权重聚类系数
- `nx.core_number(G_undirected)` 返回每个节点的核心数
- `nx.average_clustering(G_undirected, weight='weight')` 平均聚类系数

In [ ]:
# 计算度分布
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

# 转换为无向图（合并多重边为权重）
G_undirected = nx.Graph()
for u, v, d in G.edges(data=True):
    if G_undirected.has_edge(u, v):
        G_undirected[u][v]['weight'] += 1
    else:
        G_undirected.add_edge(u, v, weight=1)

# 计算聚类系数（带权重）
clustering = nx.clustering(G_undirected, weight='weight')
avg_clustering = nx.average_clustering(G_undirected, weight='weight')

# 计算核心-边缘结构
core_numbers = nx.core_number(G_undirected)

# 输出结果
print(f'Degree distribution:')
print(f'  In-degree:  mean={np.mean(list(in_degrees.values())):.2f}, max={max(in_degrees.values())}, min={min(in_degrees.values())}')
print(f'  Out-degree: mean={np.mean(list(out_degrees.values())):.2f}, max={max(out_degrees.values())}, min={min(out_degrees.values())}')
print(f'\nClustering coefficient:')
print(f'  Average: {avg_clustering:.4f}')
sorted_clustering = sorted(clustering.items(), key=lambda x: x[1], reverse=True)
print(f'  Top 5 nodes by clustering:')
for node, cc in sorted_clustering[:5]:
    print(f'    {node}: {cc:.4f} (type={node_types[node]})')
max_core = max(core_numbers.values())
core_nodes = [n for n, k in core_numbers.items() if k >= max_core]
periphery_nodes = [n for n, k in core_numbers.items() if k < max_core]
print(f'\nCore-periphery structure:')
print(f'  Max core number: {max_core}')
print(f'  Core ({len(core_nodes)} nodes): {core_nodes}')
print(f'  Periphery ({len(periphery_nodes)} nodes): {periphery_nodes}')


## 3. 多归属与锁定度：平台竞争的关键变量

**多归属（Multi-homing）**：参与者同时使用多个平台。多归属率越高，平台锁定能力越弱。

| 概念 | 定义 | 战略含义 |
|------|------|---------|
| 多归属率 | 同时使用>1个平台的参与者占比 | 高=平台锁定弱，低=锁定强 |
| 锁定度 | 1 - 1/平台数 | 高=被锁定，低=可自由迁移 |
| 平台多归属压力 | 该平台上参与者的平均平台数 | 高=参与者易流失 |

**为什么重要**：多归属是打破赢者通吃的主要力量。如果开发者可以同时在 App Store 和 Google Play 发布应用，
两个平台就难以形成垄断。这就是为什么平台会通过独家协议、排他条款来限制多归属。

## TODO 3：多归属率与锁定度分析（pandas）

**核心任务**：用 pandas 量化平台生态中的多归属率和锁定度。
- 遍历所有非平台参与者，统计其连接的平台数
- 构建 DataFrame，包含 type/platforms/multi_homing/lock_in_index 列
- 计算整体多归属率和按类型的分组多归属率
- 计算每个平台的多归属压力（该平台参与者的平均平台数）

**提示**：
- 遍历 G.nodes()，对 developer/consumer/complementor 类型节点统计其连接的平台
- `df_mh['multi_homing'] = df_mh['platforms'] > 1`
- `lock_in_index = 1.0 - 1.0 / platforms`（platforms >= 1）
- 按平台分组：筛选连接到该平台的所有参与者，计算平均平台数

In [ ]:
# 遍历非平台参与者，统计其连接的平台数
participants = {}
for node in G.nodes():
    ntype = node_types[node]
    if ntype in ('developer', 'consumer', 'complementor'):
        platforms_used = set()
        for _, target, d in G.out_edges(node, data=True):
            if d.get('relation') in ('PUBLISHES_ON', 'USES', 'INTEGRATES_WITH'):
                if node_types[target] == 'platform':
                    platforms_used.add(target)
        participants[node] = {
            'type': ntype,
            'platforms': len(platforms_used),
            'platform_list': sorted(platforms_used),
        }

df_mh = pd.DataFrame.from_dict(participants, orient='index')
df_mh['multi_homing'] = df_mh['platforms'] > 1
df_mh['lock_in_index'] = 1.0 - 1.0 / df_mh['platforms'].clip(lower=1)

# 输出结果
print(f'Multi-homing analysis ({len(df_mh)} participants):')
print(df_mh[['type', 'platforms', 'multi_homing']].to_string())
multi_homing_rate = df_mh['multi_homing'].mean()
print(f'\nOverall multi-homing rate: {multi_homing_rate:.1%}')
print(f'\nMulti-homing rate by participant type:')
for ptype in df_mh['type'].unique():
    subset = df_mh[df_mh['type'] == ptype]
    print(f'  {ptype}: {subset["multi_homing"].mean():.1%} ({subset["multi_homing"].sum()}/{len(subset)})')
print(f'\nLock-in index (higher = more locked in):')
print(df_mh[['type', 'platforms', 'lock_in_index']].to_string())
print(f'\nPlatform multi-homing pressure:')
for platform in [n for n in G.nodes() if node_types[n] == 'platform']:
    p_on = df_mh[df_mh['platform_list'].apply(lambda x: platform in x)]
    avg_plat = p_on['platforms'].mean() if len(p_on) > 0 else 0
    print(f'  {platform}: {avg_plat:.2f}')


## 4. 生态网络可视化

一张图胜过千行数字。可视化让平台生态的拓扑结构一目了然：

- **左面板**：完整生态网络，节点按类型着色（红=平台，蓝=开发者，绿=消费者，橙=互补者）
- **右面板**：核心-边缘结构，红色=核心节点（大），蓝色=边缘节点（小）

可视化能直观回答：谁是生态的枢纽？谁是可以被替代的边缘参与者？


## TODO 4：生态网络可视化（matplotlib）

**核心任务**：用 matplotlib + networkx 绘制双面板生态网络图。
- 左面板：完整网络，节点按类型着色，平台节点加大
- 右面板：核心-边缘结构，核心节点红色大，边缘节点蓝色小
- 用 spring_layout 布局，seed=42 保证可复现

**提示**：
- `fig, axes = plt.subplots(1, 2, figsize=(16, 7))`
- `pos = nx.spring_layout(G_undirected, k=2.5, seed=42, iterations=100)`
- `nx.draw_networkx_nodes/edges/labels` 绑定 ax 参数
- `plt.savefig('/tmp/s4d4_ecosystem_viz.png', dpi=150, bbox_inches='tight')`

In [ ]:
# 绘制双面板生态网络图
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 颜色映射
color_map = {
    'platform': '#E63946',
    'developer': '#457B9D',
    'consumer': '#2A9D8F',
    'complementor': '#F4A261',
}
node_colors = [color_map.get(node_types[n], '#999999') for n in G_undirected.nodes()]
node_sizes = []
for n in G_undirected.nodes():
    if node_types[n] == 'platform':
        node_sizes.append(2000)
    else:
        node_sizes.append(800)

# 布局
pos = nx.spring_layout(G_undirected, k=2.5, seed=42, iterations=100)
labels = {n: n.replace('_', ' ') for n in G_undirected.nodes()}

# 左面板：完整生态网络
ax1 = axes[0]
nx.draw_networkx_nodes(G_undirected, pos, ax=ax1, node_color=node_colors,
                       node_size=node_sizes, alpha=0.85, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(G_undirected, pos, ax=ax1, alpha=0.3, width=1.5)
nx.draw_networkx_labels(G_undirected, pos, labels=labels, ax=ax1, font_size=7, font_weight='bold')
ax1.set_title('Platform Ecosystem Network\n(Red=Platform, Blue=Developer, Green=Consumer, Orange=Complementor)',
              fontsize=10, fontweight='bold')
ax1.axis('off')

# 右面板：核心-边缘结构
ax2 = axes[1]
core_color = '#E63946'
periphery_color = '#A8DADC'
cp_colors = [core_color if n in core_nodes else periphery_color for n in G_undirected.nodes()]
cp_sizes = [1500 if n in core_nodes else 500 for n in G_undirected.nodes()]
nx.draw_networkx_nodes(G_undirected, pos, ax=ax2, node_color=cp_colors,
                       node_size=cp_sizes, alpha=0.85, edgecolors='black', linewidths=1.5)
nx.draw_networkx_edges(G_undirected, pos, ax=ax2, alpha=0.2, width=1.0)
nx.draw_networkx_labels(G_undirected, pos, labels=labels, ax=ax2, font_size=7, font_weight='bold')
ax2.set_title(f'Core-Periphery Structure\n(Core={len(core_nodes)}, Periphery={len(periphery_nodes)})',
              fontsize=10, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.savefig('/tmp/s4d4_ecosystem_viz.png', dpi=150, bbox_inches='tight')
plt.close()

print(f'Visualization saved to /tmp/s4d4_ecosystem_viz.png')
print(f'  Left panel: Full ecosystem with {G_undirected.number_of_nodes()} nodes, {G_undirected.number_of_edges()} edges')
print(f'  Right panel: Core ({len(core_nodes)}) vs Periphery ({len(periphery_nodes)})')


## 5. 平台战略框架：网络效应强度与赢者通吃倾向

不同平台有不同的战略定位。用 pandas 结构化对比4个真实平台：

| 维度 | 计算方式 | 含义 |
|------|---------|------|
| 网络效应强度 | log10(生态规模) / max(log10) | 规模越大，网络效应越强 |
| 赢者通吃倾向 | 抽成*0.3 + 网络效应*0.5 + (1-开放度)*0.2 | 高=更可能WTA |

**框架来源**：Platform Revolution（Parker et al.）+ 数据网络效应理论。
AI 平台的特殊性在于数据飞轮--这使 Hugging Face 虽然0抽成，但仍有强大的网络效应。

## TODO 5：平台战略框架分析（pandas）

**核心任务**：用 pandas 构建平台战略对比框架。
- 创建 DataFrame，包含4个平台的 commission/apps_or_models/network_effect_type/launched_year/openness
- 计算网络效应强度：log10(生态规模)归一化
- 计算赢者通吃倾向：抽成*0.3 + 网络效应*0.5 + (1-开放度)*0.2
- 打印平台定位矩阵

**提示**：
- `np.log10(apps_or_models)` 计算对数生态规模
- 开放度：Closed=0, Open(walled garden)=0.5, Open Source=1, Open Protocol=1
- WTA = commission_rate * 0.3 + network_effect_strength * 0.5 + (1 - openness_score) * 0.2

In [ ]:
# 构建平台战略对比 DataFrame
platform_strategy = pd.DataFrame({
    'platform': ['App Store', 'Google Play', 'Hugging Face', 'MCP Ecosystem'],
    'commission_rate': [0.30, 0.30, 0.00, 0.00],
    'apps_or_models': [1_800_000, 2_500_000, 1_000_000, 5000],
    'network_effect_type': ['Traditional (user count)', 'Traditional (user count)',
                            'Data network effect', 'Tool ecosystem effect'],
    'launched_year': [2008, 2008, 2016, 2024],
    'openness': ['Closed', 'Open (walled garden)', 'Open Source', 'Open Protocol'],
})

# 计算网络效应强度：log10 生态规模归一化
platform_strategy['log_ecosystem_size'] = np.log10(platform_strategy['apps_or_models'])
platform_strategy['network_effect_strength'] = (
    platform_strategy['log_ecosystem_size'] / platform_strategy['log_ecosystem_size'].max()
)

# 开放度评分
openness_map = {'Closed': 0.0, 'Open (walled garden)': 0.5, 'Open Source': 1.0, 'Open Protocol': 1.0}
platform_strategy['openness_score'] = platform_strategy['openness'].map(openness_map)

# 计算赢者通吃倾向
platform_strategy['wta_tendency'] = (
    platform_strategy['commission_rate'] * 0.3
    + platform_strategy['network_effect_strength'] * 0.5
    + (1 - platform_strategy['openness_score']) * 0.2
)

# 输出结果
print('Platform Strategy Comparison:')
print(platform_strategy.to_string(index=False))
print(f'\nNetwork effect strength (normalized, 0-1):')
for _, row in platform_strategy.iterrows():
    print(f"  {row['platform']}: {row['network_effect_strength']:.3f} (ecosystem size: {row['apps_or_models']:,})")
print(f'\nWinner-take-all tendency (0-1):')
for _, row in platform_strategy.sort_values('wta_tendency', ascending=False).iterrows():
    print(f"  {row['platform']}: {row['wta_tendency']:.3f}")


## 6. 天道推演：平台临界点蒙特卡洛模拟

> **天道推演**：以天神视角俯视平台竞争格局，在意识中构建无限可能的沙盘，
> 模拟不同网络效应强度下的平台临界点，从中选择最优策略或预判风险。

**模型**：两个竞争平台（A vs B），初始50-50市场份额。每步：
- 采纳概率 = base_rate + network_coeff * current_share（网络效应）
- 加入随机噪声（市场不确定性）
- 当份额超过临界阈值时，平台倾覆（tipping）走向赢者通吃

**贝叶斯先验**：
- 临界阈值 ~ Beta(8, 3)，均值~0.73（平台在73%份额时倾覆）
- 网络效应系数 ~ Normal(0.015, 0.005)
- 基础采纳率 ~ Uniform(0.01, 0.03)

**推演输出**：倾覆概率 / 平均倾覆步数 / 最终份额分布 / 战略启示

**2026前沿连接**：MCP/A2A 生态是新型平台形态，多Agent仿真是推演平台演化的高级方法。

## TODO 6：天道推演--平台临界点蒙特卡洛模拟（numpy + 贝叶斯）

**核心任务**：用 numpy 蒙特卡洛模拟平台竞争的临界点。
- 1000次模拟，每次50步
- 贝叶斯先验采样：阈值~Beta(8,3)，网络系数~Normal(0.015,0.005)，基础率~Uniform(0.01,0.03)
- 每步更新份额：share_A += (adopt_A - adopt_B) + noise
- 当 share_A > threshold 或 share_A < (1-threshold) 时，平台倾覆
- 统计：倾覆率、平均倾覆步数、最终份额分布
- 打印天道推演战略启示

**提示**：
- `np.random.beta(8, 3)` 采样临界阈值
- `np.random.normal(0.015, 0.005)` 采样网络效应系数
- `np.clip(share_A, 0.01, 0.99)` 防止份额越界
- 绘制最终份额分布直方图

In [ ]:
# 天道推演：平台临界点蒙特卡洛模拟
np.random.seed(42)
n_simulations = 1000
n_steps = 50
tipping_points = []

for sim in range(n_simulations):
    # 贝叶斯先验采样
    tipping_threshold = np.random.beta(8, 3)     # 均值~0.73，平台在73%份额时倾覆
    network_coeff = np.random.normal(0.015, 0.005)  # 网络效应强度
    base_rate_A = np.random.uniform(0.01, 0.03)  # 平台A基础采纳率
    base_rate_B = np.random.uniform(0.01, 0.03)  # 平台B基础采纳率

    share_A = 0.5  # 初始50-50
    tipped = False
    tip_step = n_steps

    for step in range(n_steps):
        # 采纳概率（含网络效应）
        adopt_A = base_rate_A + network_coeff * share_A
        adopt_B = base_rate_B + network_coeff * (1 - share_A)

        # 加入噪声并更新份额
        noise = np.random.normal(0, 0.015)
        share_A += (adopt_A - adopt_B) + noise
        share_A = np.clip(share_A, 0.01, 0.99)

        # 检查倾覆条件
        if not tipped and (share_A > tipping_threshold or share_A < (1 - tipping_threshold)):
            tipped = True
            tip_step = step
            break

    tipping_points.append({
        'tipped': tipped, 'tip_step': tip_step,
        'tipping_threshold': tipping_threshold,
        'network_coeff': network_coeff,
        'final_share_A': share_A,
    })

df_sim = pd.DataFrame(tipping_points)

# 输出结果
print(f'Monte Carlo Simulation: {n_simulations} runs, {n_steps} steps each')
print(f'Tipping occurred in {df_sim["tipped"].mean():.1%} of simulations')
tipped_sims = df_sim[df_sim['tipped']]
print(f'Average tipping step: {tipped_sims["tip_step"].mean():.1f} (out of {n_steps})')
print(f'\nBayesian posterior on tipping threshold:')
print(f'  Prior: Beta(8, 3), mean~0.73')
print(f'  Posterior mean (tipped sims): {tipped_sims["tipping_threshold"].mean():.3f}')
print(f'  Posterior std:  {tipped_sims["tipping_threshold"].std():.3f}')
print(f'\nNetwork effect coefficient: mean={df_sim["network_coeff"].mean():.4f}, std={df_sim["network_coeff"].std():.4f}')
print(f'Final platform A share: mean={df_sim["final_share_A"].mean():.3f}, std={df_sim["final_share_A"].std():.3f}, median={df_sim["final_share_A"].median():.3f}')
print(f'\n天道推演 Strategic Implications:')
print(f'  1. Platform tipping is probabilistic, not deterministic')
print(f'  2. Average tipping occurs at step {tipped_sims["tip_step"].mean():.0f} -- early mover advantage is real but not guaranteed')
print(f'  3. Network effect coefficient (mean={df_sim["network_coeff"].mean():.4f}) determines speed of tipping')
print(f'  4. {df_sim["tipped"].mean():.0%} of simulations reach tipping -- most platform wars do end in WTA')
print(f'  5. Multi-homing (TODO3) is the primary defense against tipping')

# 绘制分布图
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.hist(df_sim['final_share_A'], bins=50, edgecolor='black', alpha=0.7, color='#457B9D')
ax.axvline(x=0.5, color='red', linestyle='--', label='50-50 split')
ax.axvline(x=df_sim['final_share_A'].mean(), color='orange', linestyle='-',
           label=f'Mean={df_sim["final_share_A"].mean():.3f}')
ax.set_xlabel('Platform A Final Market Share')
ax.set_ylabel('Frequency')
ax.set_title(f'天道推演: Platform Tipping Point Distribution\n({n_simulations} Monte Carlo simulations, Bayesian priors)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/s4d4_tipping_point.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'\nTipping point distribution plot saved to /tmp/s4d4_tipping_point.png')


## 总结与下一步

### 今日核心发现
1. **生态网络拓扑**：平台生态有明确的核心-边缘结构，平台和核心参与者占据枢纽位置
2. **多归属率**：不同参与者类型的多归属率差异显著（消费者>开发者>互补者），决定了平台的锁定能力
3. **战略定位**：抽成比例+开放度+网络效应强度共同决定赢者通吃倾向
4. **天道推演**：平台倾覆是概率事件，网络效应系数和临界阈值共同决定倾覆速度

### 天道推演的核心启示
- 平台竞争不是注定的，而是一棵概率树
- 多归属是打破赢者通吃的主要力量
- 数据网络效应（AI平台特有）比传统网络效应更难被打破
- MCP/A2A 开放协议代表去中心化平台新范式，可能颠覆传统抽成模式

### 下一步
- **Day 5**：商业模式画布 + ROI评估--将平台战略分析纳入投资决策框架
- **跨技能**：用多Agent仿真（Mesa）建模平台生态中的自主Agent参与者
- **实践**：选择一个真实平台（如 Hugging Face vs 新进入者），用天道推演框架分析其竞争格局